Your notebook implements a **full ETL pipeline**, and inside the **Transformation phase (`etl_transform_v1`, `etl_transform_v2`, validation)** you used several **PySpark DataFrame transformations**.

Below is a **clean categorized list of the transformations used in your code**.

---

# 1. Column Selection

### `select()`

Used to select specific columns.

Example in your code:

```python
subcategories_df = categories_df.select(
    "sub_category_id", "sub_category_name", "category_id"
)
```

---

# 2. Column Creation / Modification

### `withColumn()`

Used many times to create new columns.

Examples:

```python
.withColumn("full_name", concat(col("first_name"), col("last_name")))
```

```python
.withColumn("domain", split("email", "@")[1])
```

```python
.withColumn("price_range",
    when(col("price") > col("avg_price"), "High")
    .when(col("price") == col("avg_price"), "Medium")
    .otherwise("Low")
)
```

```python
.withColumn("gross_total",
    round(getTotalCostUdf(col("product_id"), col("quantity")), 2)
)
```

---

# 3. Column Renaming

### `withColumnRenamed()`

Example:

```python
.withColumnRenamed("count", "no_of_returns")
```

---

# 4. Removing Columns

### `drop()`

Example:

```python
categories_df.drop("sub_category_id", "sub_category_name")
```

```python
.drop("order_id_surrogate", "campaign_id")
```

---

# 5. Removing Duplicate Records

### `distinct()`

Example:

```python
customers_df.distinct()
```

```python
categories_df.distinct()
```

---

# 6. Sorting Data

### `orderBy()`

Example:

```python
.orderBy("category_id")
```

```python
.orderBy("product_id")
```

---

# 7. Aggregations

### `groupBy()`

Used to aggregate data.

Example:

```python
avg_ratings_df = products_ratings_df.groupBy("product_id").agg(...)
```

---

# 8. Aggregate Functions

Used inside `agg()`:

Functions used:

```
avg()
sum()
count()
max()
min()
round()
```

Example:

```python
round(avg("ratings"), 2)
count("product_id")
sum("quantity")
```

---

# 9. Joins

### `join()`

Example:

```python
products_ratings_df.join(avg_ratings_df, "product_id", "left")
```

```python
fact_v1.join(order_items_df, "product_id", "left")
```

---

# 10. Handling Null Values

### `fillna()`

Example:

```python
products_ratings_df.fillna({"avg_rating": 0})
```

```python
customers_df.fillna(0).fillna('Unknown')
```

---

# 11. String Transformations

Functions used:

```
concat()
split()
trim()
lower()
```

Examples:

```python
concat(col("first_name"), col("last_name"))
```

```python
split("email", "@")
```

```python
lower(trim("review"))
```

---

# 12. Conditional Logic

### `when()`

Example:

```python
when(col("price") > col("avg_price"), "High")
```

---

# 13. Date Transformations

Functions used:

```
date_format()
year()
month()
weekday()
```

Example:

```python
.withColumn("year", year("order_date"))
```

---

# 14. UDF (User Defined Function)

You created two UDFs:

```python
@udf(FloatType())
def getTotalCostUdf(product_id, qty)
```

```python
@udf(FloatType())
def getTotalDiscountUdf(product_id, qty, discount)
```

Used inside:

```python
.withColumn("gross_total", getTotalCostUdf(...))
```

---

# 15. SQL Based Transformation

You also used **Spark SQL**:

```python
spark.sql("SELECT * FROM delta.`dbfs:/dbfs/ECOMMERCE_DB/products_v1`")
```

---

# Complete Transformation List Used in Your Project

| Category            | Functions                                                |
| ------------------- | -------------------------------------------------------- |
| Column selection    | `select()`                                               |
| Column creation     | `withColumn()`                                           |
| Rename column       | `withColumnRenamed()`                                    |
| Remove column       | `drop()`                                                 |
| Remove duplicates   | `distinct()`                                             |
| Sorting             | `orderBy()`                                              |
| Aggregation         | `groupBy()`                                              |
| Aggregate functions | `avg()`, `sum()`, `count()`, `max()`, `min()`, `round()` |
| Join                | `join()`                                                 |
| Null handling       | `fillna()`                                               |
| String processing   | `concat()`, `split()`, `trim()`, `lower()`               |
| Conditional logic   | `when()`                                                 |
| Date extraction     | `year()`, `month()`, `weekday()`, `date_format()`        |
| Custom logic        | `UDF`                                                    |
| SQL transformations | `spark.sql()`                                            |

---

✅ **Total transformations used:** ~15–18 core Spark transformations.

---

If you want, I can also show you **how a senior data engineer would explain this pipeline in an interview (very powerful answer)**.


### Databricks Secrets – Quick Steps

**1️⃣ Create Secret Scope**

```bash
databricks secrets create-scope --scope db-secrets
```

---

**2️⃣ Add Secrets (Keys)**

Add database password:

```bash
databricks secrets put --scope db-secrets --key db-password
```

Add database user:

```bash
databricks secrets put --scope db-secrets --key db-user
```

You will be prompted to enter the value.

---

**3️⃣ Verify Keys**

```bash
databricks secrets list --scope db-secrets
```

Example output:

```
Key name      Last updated
------------  -------------
db-password
db-user
```

---

**4️⃣ Use in Databricks Notebook**

```python
user = dbutils.secrets.get("db-secrets", "db-user")
password = dbutils.secrets.get("db-secrets", "db-password")
```

---

✔ **Scope → container for secrets**
✔ **Key → actual secret (password, token, etc.)**



In [0]:


# read data from DB 

jdbc_url = "jdbc:postgresql://razz-db-therazz007-878c.b.aivencloud.com:22917/defaultdb"

# JDBC connection properties
# driver  -> JDBC driver required for PostgreSQL
# user    -> database username
# password-> database password
properties = {
     "driver": "org.postgresql.Driver",
    "user": dbutils.secrets.get(scope="db-secrets",key="db-user")   ,
    "password": dbutils.secrets.get(scope="db-secrets",key="db-password"),
    "url":jdbc_url, 
    "dbTable":"orders"
}

df = (spark.read
    .format("jdbc")
    .options(**properties)
    .load()    
    )


df.show()

In [0]:
# Select Statement

'''
1. SELECT()

'''


###

from pyspark.sql import functions as F 


# 1
df.select("order_id").show() 


#2 
df.select(F.col("order_id")).show() 



# 3
df.select("*").show() 


# 4 
df.select("order_id","customer_name","product_name").show() 


# 5
df.select(F.col("order_id"),F.col("customer_name"),F.col("product_name")).show()


In [0]:
'''
2. selectExpr() Statmenets


'''
df.show() 
# select with operations 

df.selectExpr("order_id","(price * 10) as new_price ").show() 

df.select(F.round(F.avg("price"))).show()


In [0]:
# withColumn(colName,value)


# withColumn used to add new columns or modify existing columns 



df.withColumn("first_name",F.split(F.col("customer_name")," ").getItem(0))\
    .withColumn("last_name",F.split(F.col("customer_name")," ").getItem(1)).show() 

In [0]:

## OrderBy vs Sort 

from pyspark.sql import functions as F 

#df.sort("customer_name",ascending=True).show() 


#df.orderBy("customer_name",ascending=True).show()


# sort column wise 

df.sort(F.col("customer_name").asc(),F.col("order_id").desc()).show()


In [0]:
# groupBy() clause 



df.groupBy("product_category").count().withColumnRenamed("count","counts").show() 



df.groupBy("product_category","order_date")

In [0]:
# Problem 1 — Total Quantity by Product Category




df.groupBy("product_category").sum("quantity").show() 

In [0]:
df.groupBy("country").sum("price").withColumnRenamed("sum(price)","total_price").orderBy("total_price").show() # Problem 2 — Top 5 Countries by Revenue




df.groupBy("country").sum("price").withColumnRenamed("sum(price)","total_price").orderBy(F.desc("total_price")).limit(5).show() 

In [0]:
df.groupBy("payment_method").agg(
    F.round(F.avg("price"),2).alias("avg_price")
).show() 

In [0]:
df.groupBy("country","product_category").count().withColumnRenamed("count","counts").orderBy("country",F.desc("counts")).show()